# Satellite image exploration and SAM 3 inference

Select **Python (sat_clas)**, then **Restart Kernel → Run All**.
The default run downloads a 12.7 MB Berkeley example once, loads a geographic
image window, segments buildings with your existing local SAM 3 weights,
displays the outlines, and saves predictions. No model weights are downloaded.

To use your own imagery, change only the **Settings** cell. This notebook
processes one image/window; it does not train a model or process an entire city.

Demo source: [SamGeo's SAM 3 remote-sensing example](https://samgeo.gishub.org/examples/sam3_image_segmentation/),
using [uc_berkeley.tif](https://huggingface.co/datasets/giswqs/geospatial/resolve/main/uc_berkeley.tif).

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import rasterio
from rasterio.enums import ColorInterp
from rasterio.windows import Window
from PIL import Image, ImageOps
from IPython.display import display

from src.utils.configuration import PROJECT_ROOT, load_config
from src.data.raster_loading import read_raster
from src.data.preprocessing import rgb_preview
from src.models.sam3_loader import find_snapshot, inspect_snapshot, load_sam3

print("Project:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select the Python (sat_clas) kernel with CUDA support.")
print("GPU:", torch.cuda.get_device_name(0))

## 1. Settings

Leave `IMAGE_PATH = None` to run the Berkeley example immediately.
For your own file, set a path such as `PROJECT_ROOT / "data/raw/scene.tif"`.
GeoTIFF and ordinary PNG/JPEG images are supported. A PNG/JPEG has no map
coordinates in this workflow.

GeoTIFF RGB bands are inferred only when red/green/blue bands are declared
in the metadata. Otherwise set `RGB_BANDS` explicitly from the provider's
band definitions. `WINDOW_ORIGIN = None` selects a centered crop; an explicit
origin is `(column_offset, row_offset)` in pixels. The model prompt is a
short visible concept such as `building`, `tree`, or `swimming pool`.

In [ ]:
IMAGE_PATH = None
RGB_BANDS = None  # Example: (1, 2, 3), but confirm your raster's band order.
WINDOW_SIZE = 1008
WINDOW_ORIGIN = None
PROMPT = "building"
SCORE_THRESHOLD = 0.5
MASK_THRESHOLD = 0.5
OUTPUT_DIR = PROJECT_ROOT / "outputs/predictions/notebook_demo"

# Reset results when settings change, so old images/results are not reused.
image = pixels = profile = valid_pixels = result = masks = prediction_table = None
boxes = scores = figure = source_path = source_metadata = None

## 2. Load the image and inspect its metadata

This cell always loads a valid image or gives a specific input error before
allocating the model. It preserves the GeoTIFF window's CRS and transform.
Three-band uint8 imagery is used directly; other raster values receive a
per-band percentile stretch for this visual demonstration. Production
preprocessing should be fixed and evaluated for the chosen imagery source.

In [ ]:
# Clear the previous image and predictions before attempting another input.
image = pixels = profile = valid_pixels = result = masks = prediction_table = figure = None
if not isinstance(WINDOW_SIZE, int) or WINDOW_SIZE < 1:
    raise ValueError("WINDOW_SIZE must be a positive integer.")
if not isinstance(PROMPT, str) or not PROMPT.strip():
    raise ValueError("PROMPT must contain a visible object description.")
if not (0 <= SCORE_THRESHOLD <= 1 and 0 <= MASK_THRESHOLD <= 1):
    raise ValueError("Both thresholds must be between 0 and 1.")

using_demo = IMAGE_PATH is None
if using_demo:
    import requests

    sample_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/uc_berkeley.tif"
    source_path = PROJECT_ROOT / "data/raw/examples/uc_berkeley.tif"
    source_path.parent.mkdir(parents=True, exist_ok=True)
    if not source_path.is_file():
        print("Downloading the 12.7 MB Berkeley example once...")
        response = requests.get(sample_url, timeout=(15, 120))
        response.raise_for_status()
        partial_path = source_path.with_suffix(".tif.part")
        partial_path.write_bytes(response.content)
        partial_path.replace(source_path)
    else:
        print("Using the cached Berkeley example.")
else:
    source_path = Path(IMAGE_PATH).expanduser()
    if not source_path.is_absolute():
        source_path = PROJECT_ROOT / source_path
    if not source_path.is_file():
        raise FileNotFoundError(f"Image does not exist: {source_path}. Set IMAGE_PATH or use None for the demo.")

source_metadata = {"source_path": str(source_path), "demo": using_demo}
if using_demo:
    source_metadata["source_url"] = sample_url

if source_path.suffix.lower() in {".tif", ".tiff"}:
    with rasterio.open(source_path) as dataset:
        print("Raster:", dataset.width, "x", dataset.height, "pixels; bands:", dataset.count)
        print("Band types:", dataset.dtypes)
        print("Color interpretation:", [value.name for value in dataset.colorinterp])
        print("CRS:", dataset.crs)
        selected_bands = RGB_BANDS
        if selected_bands is None:
            rgb_colors = (ColorInterp.red, ColorInterp.green, ColorInterp.blue)
            if all(color in dataset.colorinterp for color in rgb_colors):
                selected_bands = tuple(dataset.colorinterp.index(color) + 1 for color in rgb_colors)
            elif using_demo and dataset.count == 3:
                selected_bands = (1, 2, 3)  # This published example is RGB.
            else:
                raise ValueError("RGB band order is not declared. Set RGB_BANDS from the imagery provider's metadata.")
        if len(selected_bands) != 3:
            raise ValueError("RGB_BANDS must contain exactly three one-based indexes.")
        width, height = min(WINDOW_SIZE, dataset.width), min(WINDOW_SIZE, dataset.height)
        if WINDOW_ORIGIN is None:
            col, row = (dataset.width - width) // 2, (dataset.height - height) // 2
        else:
            col, row = WINDOW_ORIGIN
            if col != int(col) or row != int(row) or not (0 <= col < dataset.width and 0 <= row < dataset.height):
                raise ValueError("WINDOW_ORIGIN must be integer offsets inside the image.")
        window = Window(col, row, width, height)

    pixels, profile = read_raster(source_path, bands=selected_bands, window=window)
    valid_pixels = ~np.ma.getmaskarray(pixels).any(axis=0)
    valid_pixels &= np.isfinite(pixels.data).all(axis=0)
    if not valid_pixels.any():
        raise ValueError("The selected window contains only nodata. Choose another WINDOW_ORIGIN.")
    if pixels.dtype == np.uint8:
        rgb = pixels.filled(0).transpose(1, 2, 0).copy()
        rgb[~valid_pixels] = 0
        image = Image.fromarray(rgb)
        preprocessing = "uint8 RGB; no percentile stretch"
    else:
        image = rgb_preview(pixels)
        preprocessing = "per-band 2nd/98th percentile stretch"
    source_metadata.update({
        "rgb_bands": list(selected_bands), "window_origin": [int(col), int(row)],
        "crs": str(profile["crs"]) if profile["crs"] else None,
        "transform": list(profile["transform"])[:6], "preprocessing": preprocessing,
    })
elif source_path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
    with Image.open(source_path) as opened:
        original_image = ImageOps.exif_transpose(opened).convert("RGB")
        width, height = min(WINDOW_SIZE, original_image.width), min(WINDOW_SIZE, original_image.height)
        if WINDOW_ORIGIN is None:
            col, row = (original_image.width - width) // 2, (original_image.height - height) // 2
        else:
            col, row = WINDOW_ORIGIN
            if col != int(col) or row != int(row) or not (0 <= col < original_image.width and 0 <= row < original_image.height):
                raise ValueError("WINDOW_ORIGIN must be integer offsets inside the image.")
        image = original_image.crop((col, row, min(col + width, original_image.width), min(row + height, original_image.height)))
    valid_pixels = np.ones((image.height, image.width), dtype=bool)
    source_metadata.update({"crs": None, "window_origin": [int(col), int(row)], "preprocessing": "RGB conversion"})
else:
    raise ValueError("Use a GeoTIFF (.tif/.tiff), PNG, or JPEG image.")

source_metadata["image_size"] = [image.width, image.height]
print("Loaded:", source_path.name, "| inference window:", image.size)
display(image.resize((min(700, image.width), max(1, round(image.height * min(700, image.width) / image.width)))))

## 3. Load your local SAM 3 model

Model loading happens after image validation. Re-running with another prompt
reuses the loaded model. To release GPU memory completely, restart the kernel.

In [ ]:
if globals().get("image") is None:
    raise RuntimeError("Run the image-loading cell successfully before loading SAM 3.")

sam_settings = load_config()["sam3"]
cache_report = inspect_snapshot(find_snapshot(sam_settings))
if not cache_report["complete"]:
    raise FileNotFoundError(f"SAM 3 cache is incomplete: {cache_report['missing']}")
requested_model_key = json.dumps(sam_settings, sort_keys=True)
if globals().get("_loaded_sam3_key") != requested_model_key or "model" not in globals():
    model, processor = load_sam3(sam_settings)
    _loaded_sam3_key = requested_model_key
else:
    print("Reusing the loaded SAM 3 model.")
print("SAM 3:", model.device, "|", model.dtype)

## 4. Segment the selected concept

Returned masks belong to this image window. Scores are model scores, not
calibrated probabilities. Nodata pixels are removed from the output masks.

In [ ]:
if globals().get("image") is None or "model" not in globals():
    raise RuntimeError("Run image loading and model loading first, or use Run All.")
result = masks = prediction_table = None
inputs = processor(images=image, text=PROMPT.strip(), return_tensors="pt").to(model.device)
torch.cuda.synchronize()
started = time.perf_counter()
with torch.inference_mode():
    outputs = model(**inputs)
torch.cuda.synchronize()
result = processor.post_process_instance_segmentation(
    outputs,
    threshold=SCORE_THRESHOLD,
    mask_threshold=MASK_THRESHOLD,
    target_sizes=inputs["original_sizes"].tolist(),
)[0]
masks = result["masks"].detach().cpu().numpy().astype(bool)
boxes = result["boxes"].detach().cpu().numpy()
scores = result["scores"].detach().cpu().numpy()
if masks.size == 0:
    masks = np.empty((0, image.height, image.width), dtype=bool)
masks &= valid_pixels[None, :, :]
keep = masks.any(axis=(1, 2))
masks, boxes, scores = masks[keep], boxes[keep], scores[keep]
inference_seconds = time.perf_counter() - started
# Release temporary GPU tensors; retain the model for the next prompt.
del inputs, outputs, result
result = {"masks": masks, "boxes": boxes, "scores": scores}
print(f"Detected {len(masks)} instances for {PROMPT!r} in {inference_seconds:.2f} s.")
if len(masks) == 0:
    print("No instances passed the thresholds. Inspect the image/prompt before adjusting the score threshold.")

## 5. Inspect outlines and scores

The image and predicted outlines are shown side by side. The table includes
pixel-space boxes and mask areas within the selected window.

In [ ]:
if globals().get("masks") is None:
    raise RuntimeError("Run the inference cell before visualizing predictions.")
figure, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(image)
axes[0].set_title("Input image window")
axes[1].imshow(image)
foreground = masks.any(axis=0)
overlay = np.zeros((image.height, image.width, 4), dtype=np.float32)
overlay[foreground] = (0.1, 1.0, 0.2, 0.25)
axes[1].imshow(overlay)
for mask in masks:
    if mask.any() and not mask.all():
        axes[1].contour(mask, levels=[0.5], colors="lime", linewidths=0.8)
axes[1].set_title(f"{PROMPT}: {len(masks)} instances")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

prediction_table = pd.DataFrame([
    {"object_id": index + 1, "class": PROMPT, "score": float(score),
     "xmin": float(box[0]), "ymin": float(box[1]),
     "xmax": float(box[2]), "ymax": float(box[3]),
     "area_pixels": int(mask.sum())}
    for index, (mask, box, score) in enumerate(zip(masks, boxes, scores))
], columns=["object_id", "class", "score", "xmin", "ymin", "xmax", "ymax", "area_pixels"])
display(prediction_table)

## 6. Save predictions

Each run gets its own directory under `OUTPUT_DIR`: an overlay PNG,
instance masks/boxes/scores in NPZ, a CSV table, and source/model metadata.
Georeferenced rasters also produce a binary mask GeoTIFF with the window's
CRS and transform: 1 = detected foreground, 0 = background, 255 = nodata.
Individual instances remain separate in the NPZ file.

In [ ]:
from datetime import datetime, timezone
import uuid

if globals().get("prediction_table") is None or globals().get("figure") is None:
    raise RuntimeError("Run inference and visualization before saving predictions.")
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = Path(OUTPUT_DIR) / run_name
run_dir.mkdir(parents=True, exist_ok=False)
figure.savefig(run_dir / "overlay.png", dpi=150, bbox_inches="tight")
np.savez_compressed(run_dir / "instances.npz", masks=masks, boxes=boxes, scores=scores)
prediction_table.to_csv(run_dir / "predictions.csv", index=False)
metadata = {
    **source_metadata, "prompt": PROMPT, "score_threshold": SCORE_THRESHOLD,
    "mask_threshold": MASK_THRESHOLD, "model_id": sam_settings["model_id"],
    "model_revision": sam_settings.get("revision"), "model_dtype": str(model.dtype),
    "count": len(masks), "inference_seconds": round(inference_seconds, 3),
}
(run_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
if profile is not None and profile.get("crs") is not None:
    binary_mask = masks.any(axis=0).astype(np.uint8)
    binary_mask[~valid_pixels] = 255
    with rasterio.open(
        run_dir / "foreground_mask.tif", "w", driver="GTiff",
        width=image.width, height=image.height, count=1, dtype="uint8",
        crs=profile["crs"], transform=profile["transform"], nodata=255, compress="deflate",
    ) as destination:
        destination.write(binary_mask, 1)
print("Saved results:", run_dir)
for saved_path in sorted(run_dir.iterdir()):
    print(" -", saved_path.name)

## Use these predictions in the map application

The next cell exports the **existing** masks to WGS84 GeoJSON, including a
geohash per object. It does not run SAM a second time. Open the map interface in
a terminal with `conda run --no-capture-output -n sat_clas python -m src.cli serve`,
then visit **http://127.0.0.1:8765** and import the exported GeoJSON in Explore.

The app also accepts addresses, common questions, coordinates, geohashes, image
uploads, and before/after GeoTIFFs. See `docs/training_and_map_workflows.md`.


In [ ]:
from src.inference.segmentation import vectorize_masks

if globals().get("masks") is None or "run_dir" not in globals():
    raise RuntimeError("Run the image, inference and export cells above first.")
map_profile = profile if profile is not None and profile.get("crs") is not None else None
map_features = vectorize_masks(masks, scores, map_profile, PROMPT)
map_output_path = run_dir / "features.geojson"
map_output_path.write_text(json.dumps(map_features, indent=2), encoding="utf-8")
print("Map export:", map_output_path)
print("Georeferenced objects:", len(map_features["features"]))
if map_profile is None:
    print("This image has no map coordinates. The image preview remains available.")


## Optional: attach map names, addresses and tags

Set `FETCH_MAP_EVIDENCE=True` to query the image window using OpenStreetMap.
This sends only the geographic area to the map provider. Names/addresses remain
source-backed candidates; missing records stay unknown. Multiple tenants can
occupy one building. This step uses the existing masks and does not train a model.


In [ ]:
FETCH_MAP_EVIDENCE = False

if FETCH_MAP_EVIDENCE:
    from rasterio.transform import array_bounds
    from rasterio.warp import transform_bounds
    from src.geo.providers import query_features
    from src.geo.matching import attach_map_evidence

    if map_profile is None:
        raise ValueError("A georeferenced image is required for map-data matching.")
    area_bounds = transform_bounds(map_profile["crs"], "EPSG:4326", *array_bounds(
        image.height, image.width, map_profile["transform"]))
    mapped_records = query_features(list(area_bounds), category="all")
    enriched_features = attach_map_evidence(map_features, mapped_records)
    enriched_path = run_dir / "features_with_map_evidence.geojson"
    enriched_path.write_text(json.dumps(enriched_features, indent=2), encoding="utf-8")
    print("Map evidence export:", enriched_path)
    display(pd.DataFrame([{
        "object_id": f["properties"]["object_id"],
        "candidate_name": f["properties"].get("candidate_name"),
        "candidate_address": f["properties"].get("candidate_address"),
        "map_candidates": len(f["properties"]["map_evidence"]),
    } for f in enriched_features["features"]]))
else:
    print("Map lookup is optional. Enable FETCH_MAP_EVIDENCE or use the app's Imagery tab.")


## Train and reload a tile classifier

SAM inference above already uses trained local weights. The following optional
workflow trains a separate **ResNet-18 RGB tile classifier** on your reviewed
labels; it does not fine-tune SAM or learn building addresses.

Create a CSV with `path,label,split,group` columns. Splits are `train`, `val`,
and optionally `test`; groups must identify separated geographic areas/scenes.
See the complete dataset example in `docs/training_and_map_workflows.md`.
No real labeled dataset has been supplied, so this section starts disabled.


In [ ]:
TRAINING_MANIFEST = None  # e.g. Path(r"D:/data/my_dataset/manifest.csv")
TRAINING_EPOCHS = 20
TRAINING_BATCH_SIZE = 16
NEW_TILE_PATH = None  # RGB PNG/JPEG to classify after training
trained_classifier = None


In [ ]:
from src.training.train_classifier import train_classifier, classify_image

if TRAINING_MANIFEST is None:
    print("Set TRAINING_MANIFEST to your labeled dataset CSV, then run this cell.")
else:
    training_output = PROJECT_ROOT / "outputs/training" / (
        "notebook_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    trained_classifier = train_classifier(
        manifest=TRAINING_MANIFEST,
        output=training_output,
        epochs=TRAINING_EPOCHS,
        batch_size=TRAINING_BATCH_SIZE,
        device="cuda",
    )
    print("Best checkpoint:", trained_classifier["checkpoint"])
    print("Evaluation:", json.dumps(trained_classifier["evaluation"], indent=2))


In [ ]:
if NEW_TILE_PATH is None:
    print("Set NEW_TILE_PATH to classify a tile after training, or use the app's Training tab.")
elif trained_classifier is None:
    raise RuntimeError("Train a classifier above before predicting a new tile.")
else:
    tile_prediction = classify_image(trained_classifier["checkpoint"], NEW_TILE_PATH)
    print(json.dumps(tile_prediction, indent=2))
